# Makeitalive - Google Colab Runner Extended

This notebook allows you to easily clone the GitHub repository, install dependencies, extract a dataset, analyze it, train the motion flow model, evaluate it with a sample animation, and download the resulting weights.

**Note:** To speed up training, make sure you are using a GPU runtime (`Runtime` -> `Change runtime type` -> `T4 GPU`).

## 1. Setup Environment

In [ ]:
!git clone https://github.com/matu1003/Makeitalive.git
%cd Makeitalive
!pip install -e .

import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from scipy.ndimage import map_coordinates
import glob
import argparse
import shutil
from google.colab import files

# Ensure the src directory is in the Python path for easy imports
sys.path.append(os.path.abspath('src'))

## 2. Dataset Collection

In [ ]:
from data.download_youtube import download_video
from data.make_dataset_video import extract_pairs_from_video

video_path = "./data/10hourslandscape.mp4"
dataset_dir = os.path.join(".", "data", "dataset_local")

print("1. Downloading video...")
download_video(
    url="https://www.youtube.com/watch?v=AKeUssuu3Is",
    output_path=video_path,
    max_height=720
)

print("\n2. Extracting dataset pairs...")
extract_pairs_from_video(
    video_path=video_path,
    output_dir=dataset_dir,
    sample_every_n_seconds=5.0,
    frame_gap=4,
    target_size=512,
    max_pairs=100,
    clean=True
)

## 3. Dataset Analysis
Before training, let's look at some samples and analyze the distribution of movements in our dataset.

In [ ]:
from data.dataset import LandscapeMotionDataset
import random

dataset = LandscapeMotionDataset(dataset_dir)
print(f"Dataset loaded with {len(dataset)} pairs.")

# Visualize random pairs
def show_random_samples(dataset, num_samples=3):
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    for i in range(num_samples):
        idx = random.randint(0, len(dataset) - 1)
        img_a, img_b, _ = dataset[idx]
        
        # Difference image
        diff = np.abs(img_a.astype(np.float32) - img_b.astype(np.float32))
        diff = (diff - diff.min()) / (diff.max() - diff.min() + 1e-8)
        
        axes[i, 0].imshow(img_a)
        axes[i, 0].set_title(f"Sample {idx} - Image A")
        axes[i, 1].imshow(img_b)
        axes[i, 1].set_title(f"Sample {idx} - Image B")
        axes[i, 2].imshow(diff)
        axes[i, 2].set_title("Normalized Difference")
        
        for ax in axes[i]:
            ax.axis('off')
    plt.tight_layout()
    plt.show()

show_random_samples(dataset)

### Distance Metrics Analysis
We calculate three metrics to understand the dissimilarity between pairs:
1. **MSE**: Pixel-wise Mean Squared Error.
2. **Histogram**: Bhattacharyya distance between color histograms (detects global color shifts).
3. **Thumbnail MAE**: Mean Absolute Error on 32x32 grayscale thumbnails (detects composition changes while ignoring noise).

In [ ]:
def mse_distance(img1, img2):
    return np.mean((img1.astype(np.float32) - img2.astype(np.float32)) ** 2)

def histogram_distance(img1, img2):
    # Convert to HSV for better color comparison
    hsv1 = cv2.cvtColor((img1 * 255).astype(np.uint8), cv2.COLOR_RGB2HSV)
    hsv2 = cv2.cvtColor((img2 * 255).astype(np.uint8), cv2.COLOR_RGB2HSV)
    
    hist1 = cv2.calcHist([hsv1], [0, 1], None, [180, 256], [0, 180, 0, 256])
    hist2 = cv2.calcHist([hsv2], [0, 1], None, [180, 256], [0, 180, 0, 256])
    
    cv2.normalize(hist1, hist1, 0, 1, cv2.NORM_MINMAX)
    cv2.normalize(hist2, hist2, 0, 1, cv2.NORM_MINMAX)
    
    return cv2.compareHist(hist1, hist2, cv2.HISTCMP_BHATTACHARYYA)

def thumbnail_distance(img1, img2, size=(32, 32)):
    t1 = cv2.resize(cv2.cvtColor((img1 * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY), size)
    t2 = cv2.resize(cv2.cvtColor((img2 * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY), size)
    return np.mean(np.abs(t1.astype(np.float32) - t2.astype(np.float32)))

print("Analyzing dataset distances...")
records = []
for i in range(len(dataset)):
    img_a, img_b, _ = dataset[i]
    records.append({
        'index': i,
        'mse': mse_distance(img_a, img_b),
        'hist': histogram_distance(img_a, img_b),
        'thumb': thumbnail_distance(img_a, img_b)
    })

def plot_distributions(records):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    metrics = ['mse', 'hist', 'thumb']
    colors = ['skyblue', 'salmon', 'lightgreen']
    
    for i, metric in enumerate(metrics):
        vals = [r[metric] for r in records]
        axes[i].hist(vals, bins=30, color=colors[i], edgecolor='black', alpha=0.7)
        axes[i].set_title(f"{metric.upper()} Distribution")
        axes[i].axvline(np.mean(vals), color='red', linestyle='--', label='Mean')
        axes[i].legend()
    
    plt.tight_layout()
    plt.show()

plot_distributions(records)

### Visualizing Outliers
Showing the most distant pairs (based on Histogram distance). These are likely cut scenes or significant changes that might need filtering.

In [ ]:
def show_outliers(dataset, records, top_n=3):
    sorted_records = sorted(records, key=lambda x: x['hist'], reverse=True)
    fig, axes = plt.subplots(top_n, 2, figsize=(10, 4 * top_n))
    
    for i in range(top_n):
        idx = sorted_records[i]['index']
        dist = sorted_records[i]['hist']
        img_a, img_b, _ = dataset[idx]
        
        axes[i, 0].imshow(img_a)
        axes[i, 0].set_title(f"Outlier Rank {i+1} (idx {idx}) - A")
        axes[i, 1].imshow(img_b)
        axes[i, 1].set_title(f"B (Dist: {dist:.4f})")
        
        for ax in axes[i]:
            ax.axis('off')
            
    plt.tight_layout()
    plt.show()

show_outliers(dataset, records)

## 4. Training the Model
Run the training loop on the dataset generated above.

In [ ]:
from motion_flow.train import train

# Create an arguments namespace equivalent to Python argparse
args = argparse.Namespace(
    data_dir=dataset_dir,
    ckpt_dir="./checkpoints",
    epochs=10,
    batch_size=8,
    lr=1e-4,
    num_workers=2
)

print("Starting training...")
train(args)

## 5. Evaluation and Inference
Load the trained model and test it by generating a sample animation.

In [ ]:
from motion_flow.model import MotionFlowUNet
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

TARGET_SIZE = 512
ckpt_dir = "./checkpoints"
CHECKPOINT_PATH = ""

# Find the latest checkpoint automatically
if os.path.exists(ckpt_dir):
    runs = sorted(glob.glob(os.path.join(ckpt_dir, "run_*")), reverse=True)
    if runs:
        latest_run = runs[0]
        model_path = os.path.join(latest_run, "model_best.pth")
        if not os.path.exists(model_path):
            model_path = os.path.join(latest_run, "model_latest.pth")
            
        if os.path.exists(model_path):
            CHECKPOINT_PATH = model_path
            print(f"✓ Checkpoint found: {CHECKPOINT_PATH}")

if CHECKPOINT_PATH:
    model = MotionFlowUNet().to(device)
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
    model.eval()
    print("Model loaded successfully.")
else:
    print("⚠️ No checkpoint found. Inference might not be accurate.")

In [ ]:
def warp_image(img, flow):
    h, w = img.shape[:2]
    flow_u = flow[0]
    flow_v = flow[1]
    
    y, x = np.mgrid[0:h, 0:w].astype(np.float32)
    dist_x = x - flow_u
    dist_y = y - flow_v
    
    warped_img = np.zeros_like(img)
    for i in range(3):
        warped_img[..., i] = map_coordinates(img[..., i], [dist_y, dist_x], order=1, mode='reflect')
        
    return warped_img

# Pick a sample image from the dataset
sample_images = glob.glob(os.path.join(dataset_dir, "*.jpg"))
if sample_images:
    test_image_path = sample_images[0]
    print(f"Testing on: {test_image_path}")
    
    img_pil = Image.open(test_image_path).convert('RGB').resize((TARGET_SIZE, TARGET_SIZE))
    img_np = np.array(img_pil).astype(np.float32) / 255.0
    img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).to(device)
    
    with torch.no_grad():
        flow_pred = model(img_tensor)
        flow_pred = F.interpolate(flow_pred, size=(TARGET_SIZE, TARGET_SIZE), mode='bilinear', align_corners=False)
        flow_np = flow_pred.squeeze(0).cpu().numpy()
    
    # Generate animation
    frames = []
    num_frames = 30
    for i in range(num_frames):
        alpha = i / num_frames
        warped = warp_image(img_np, flow_np * alpha * 10) # Multiply for visible effect
        frames.append(Image.fromarray((warped * 255).astype(np.uint8)))
    
    # Save as GIF
    gif_path = "output_animation.gif"
    frames[0].save(gif_path, save_all=True, append_images=frames[1:], duration=50, loop=0)
    print(f"Animation saved to: {gif_path}")
    
    # Provide download link for the GIF
    files.download(gif_path)
else:
    print("⚠️ No images found in dataset directory for testing.")

## 6. Export and Download Weights
Pack all training results into a ZIP archive and download it.

In [ ]:
# Compress the checkpoints directory
shutil.make_archive('makeitalive_checkpoints', 'zip', 'checkpoints')

# Trigger a download to save the weights locally
files.download('makeitalive_checkpoints.zip')